# 00 — Explore & Profile (Bronze)

**Tickets:** I-02  
**Purpose:** Initial EDA of the raw NYC Yellow Taxi data — row counts, null rates, distributions, outliers.

---

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# On Databricks `spark` is injected automatically.
# Uncomment the two lines below only when running locally outside Databricks.
# spark = SparkSession.builder.appName("nyc_taxi_bronze_eda").getOrCreate()

print(f"Spark version: {spark.version}")

## Row counts & schema

In [ ]:
# Update to match the catalog/schema written by I-01.
BRONZE_TABLE = "bronze.nyc_yellow_taxi"

df = spark.read.table(BRONZE_TABLE)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

df.printSchema()

# Visual spot-check of raw values.
display(df.limit(10))

## Null & duplicate analysis

In [ ]:
total_rows = df.count()

# --- Null counts ---
null_row = df.select(
    [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]
).collect()[0]

null_summary = spark.createDataFrame(
    [(c, int(null_row[c])) for c in df.columns],
    ["column_name", "null_count"],
).withColumn(
    "null_pct", F.round(F.col("null_count") / total_rows * 100, 2)
).orderBy(F.col("null_count").desc())

print("=== Null counts per column ===")
display(null_summary)

# --- Duplicate rows ---
distinct_rows = df.distinct().count()
duplicate_count = total_rows - distinct_rows
print(f"\nTotal rows    : {total_rows:,}")
print(f"Distinct rows : {distinct_rows:,}")
print(f"Duplicates    : {duplicate_count:,}  ({duplicate_count / total_rows * 100:.2f}%)")

## Value distributions & outliers

In [ ]:
# --- Numeric summary statistics ---
# Bronze columns may be ingested as strings; cast the known numeric fields for describe().
NUMERIC_COLS = [
    "Passenger_count", "Trip_distance",
    "Pickup_longitude", "Pickup_latitude",
    "Dropoff_longitude", "Dropoff_latitude",
    "Fare_amount", "Extra", "MTA_tax",
    "Improvement_surcharge", "Tip_amount", "Tolls_amount", "Total_amount",
]

numeric_df = df.select(
    [F.col(c).cast("double").alias(c) for c in NUMERIC_COLS if c in df.columns]
)

print("=== Numeric column statistics ===")
display(numeric_df.describe())

### Categorical value distributions

Inspect every low-cardinality field to catch unexpected codes or encoding issues in the raw data.

In [ ]:
CATEGORICAL_COLS = {
    "VendorID":          {1: "Creative Mobile Technologies", 2: "VeriFone Inc."},
    "RateCodeID":        {1: "Standard", 2: "JFK", 3: "Newark", 4: "Nassau/Westchester", 5: "Negotiated", 6: "Group ride"},
    "Store_and_fwd_flag":{},   # Y / N
    "Payment_type":      {1: "Credit card", 2: "Cash", 3: "No charge", 4: "Dispute", 5: "Unknown", 6: "Voided trip"},
}

for col_name in CATEGORICAL_COLS:
    if col_name not in df.columns:
        print(f"Column not found: {col_name}")
        continue
    print(f"\n=== {col_name} ===")
    display(
        df.groupBy(col_name)
          .count()
          .withColumn("pct", F.round(F.col("count") / total_rows * 100, 2))
          .orderBy(F.col("count").desc())
    )

### Outlier detection

Flag business-rule violations that should be treated as corrupt data in the Silver cleaning step (I-03/I-04).

In [ ]:
def cast(col_name: str) -> F.Column:
    """Cast a Bronze column to double for numeric comparisons."""
    return F.col(col_name).cast("double")

outlier_summary = df.agg(
    F.count("*").alias("total_rows"),
    # Distance issues
    F.sum(F.when(cast("Trip_distance") == 0,  1).otherwise(0)).alias("zero_distance"),
    F.sum(F.when(cast("Trip_distance") <  0,  1).otherwise(0)).alias("negative_distance"),
    F.sum(F.when(cast("Trip_distance") > 100, 1).otherwise(0)).alias("distance_over_100mi"),
    # Fare issues
    F.sum(F.when(cast("Fare_amount") <= 0, 1).otherwise(0)).alias("zero_or_neg_fare"),
    F.sum(F.when(cast("Fare_amount") >  500, 1).otherwise(0)).alias("fare_over_500"),
    # Total amount issues
    F.sum(F.when(cast("Total_amount") < 0, 1).otherwise(0)).alias("negative_total_amount"),
    # Passenger issues
    F.sum(F.when(cast("Passenger_count") == 0, 1).otherwise(0)).alias("zero_passengers"),
    F.sum(F.when(cast("Passenger_count") >  6, 1).otherwise(0)).alias("passengers_over_6"),
    # Tip on non-card payments (tip should only be auto-populated for credit card)
    F.sum(
        F.when(
            (cast("Payment_type") != 1) & (cast("Tip_amount") > 0), 1
        ).otherwise(0)
    ).alias("tip_on_non_card_payment"),
)

display(outlier_summary)

### Temporal distributions

Understand when trips occur — drives demand heatmaps (BQ-1) and is a key feature for ML fare prediction (BQ-3).  
> **Note:** Bronze datetimes may be raw strings; `to_timestamp` handles standard YYYY-MM-DD HH:mm:ss format.

In [ ]:
pickup_ts = F.to_timestamp("tpep_pickup_datetime")

temporal_df = df.withColumn("pickup_ts", pickup_ts)

# Date range
print("=== Pickup date range ===")
display(
    temporal_df.agg(
        F.min("pickup_ts").alias("earliest_pickup"),
        F.max("pickup_ts").alias("latest_pickup"),
    )
)

# Trips by hour of day
print("\n=== Trips by hour of day ===")
display(
    temporal_df.groupBy(F.hour("pickup_ts").alias("hour_of_day"))
               .count()
               .orderBy("hour_of_day")
)

# Trips by day of week (1 = Sunday … 7 = Saturday in Spark)
print("\n=== Trips by day of week ===")
display(
    temporal_df.groupBy(F.dayofweek("pickup_ts").alias("day_of_week"))
               .count()
               .orderBy("day_of_week")
)

# Trips by calendar month (useful for spotting seasonal patterns or bad data)
print("\n=== Trips by month ===")
display(
    temporal_df.groupBy(
        F.year("pickup_ts").alias("year"),
        F.month("pickup_ts").alias("month"),
    ).count().orderBy("year", "month")
)

## Findings & Assumptions

Record observations here after running the cells above. This feeds directly into I-07 (data quality log).

| # | Finding | Affected column(s) | Recommended action (I-03/I-04) |
|---|---------|-------------------|--------------------------------|
| 1 | _e.g. X% of rows have Trip_distance = 0_ | Trip_distance | Drop in Silver |
| 2 | _e.g. Fare_amount contains negative values_ | Fare_amount | Drop in Silver |
| 3 | _e.g. Passenger_count has 0-value rows_ | Passenger_count | Drop or impute in Silver |
| 4 | _e.g. tpep_pickup_datetime outside expected year range_ | tpep_pickup_datetime | Drop in Silver |
| 5 | _add more rows as needed_ | | |

### Assumptions
- Bronze table is append-only (no transforms applied by I-01).
- Tip amounts for non-credit-card payments are not captured in `Tip_amount` (cash tips excluded by design).
- `RateCodeID` values outside 1–6 and `Payment_type` values outside 1–6 are data entry errors.
- Rows where `Total_amount < 0` are voided/disputed trips and should be excluded from analytics.